## Ejercicio Block Cipher

In [1]:
from Crypto.Cipher import DES, DES3, AES
from Crypto.Random import get_random_bytes
from Crypto.Util.Padding import pad, unpad
from PIL import Image
import io


In [2]:
class DES_ECB:
    def __init__(self):
        self.key = get_random_bytes(8)  # DES usa llaves de 8 bytes (64 bits), pero solo 56 bits se usan

    def cifrar(self, mensaje):
        cipher = DES.new(self.key, DES.MODE_ECB)
        mensaje_padded = pad(mensaje.encode(), 8)  # Relleno manual a múltiplos de 8 bytes
        cifrado = cipher.encrypt(mensaje_padded)
        return cifrado

    def descifrar(self, cifrado):
        cipher = DES.new(self.key, DES.MODE_ECB)
        mensaje_padded = cipher.decrypt(cifrado)
        return unpad(mensaje_padded, 8).decode()


In [3]:
class TripleDES_CBC:
    def __init__(self):
        self.key = DES3.adjust_key_parity(get_random_bytes(24))  # 3DES usa llaves de 24 bytes (192 bits)
        self.iv = get_random_bytes(8)  # Vector de inicialización (IV) de 8 bytes

    def cifrar(self, mensaje):
        cipher = DES3.new(self.key, DES3.MODE_CBC, self.iv)
        mensaje_padded = pad(mensaje.encode(), 8)
        cifrado = cipher.encrypt(mensaje_padded)
        return cifrado, self.iv

    def descifrar(self, cifrado, iv):
        cipher = DES3.new(self.key, DES3.MODE_CBC, iv)
        mensaje_padded = cipher.decrypt(cifrado)
        return unpad(mensaje_padded, 8).decode()


In [4]:
class AES_CBC:
    def __init__(self):
        self.key = get_random_bytes(16)  # AES de 128 bits
        self.iv = get_random_bytes(16)  # IV de 16 bytes

    def cifrar(self, mensaje):
        cipher = AES.new(self.key, AES.MODE_CBC, self.iv)
        mensaje_padded = pad(mensaje, 16)
        cifrado = cipher.encrypt(mensaje_padded)
        return cifrado, self.iv

    def descifrar(self, cifrado, iv):
        cipher = AES.new(self.key, AES.MODE_CBC, iv)
        mensaje_padded = cipher.decrypt(cifrado)
        return unpad(mensaje_padded, 16)


In [5]:
class AES_ECB:
    def __init__(self):
        self.key = get_random_bytes(16)

    def cifrar(self, mensaje):
        cipher = AES.new(self.key, AES.MODE_ECB)
        mensaje_padded = pad(mensaje, 16)
        cifrado = cipher.encrypt(mensaje_padded)
        return cifrado

    def descifrar(self, cifrado):
        cipher = AES.new(self.key, AES.MODE_ECB)
        mensaje_padded = cipher.decrypt(cifrado)
        return unpad(mensaje_padded, 16)


In [6]:
def cargar_imagen(ruta):
    with open(ruta, "rb") as file:
        return file.read()

In [7]:
def guardar_imagen(data, ruta):
    with open(ruta, "wb") as file:
        file.write(data)

In [ ]:
if __name__ == "__main__":
    mensaje = "Hola, esto es un mensaje de prueba!"
    imagen_ruta = "pic.png"
    imagen = cargar_imagen(imagen_ruta)

    print("\n-- Prueba con AES (CBC) para Imagen --")
    aes_cbc = AES_CBC()
    cifrado_imagen_cbc, iv_cbc = aes_cbc.cifrar(imagen)
    guardar_imagen(cifrado_imagen_cbc, "imagen_cifrada_cbc.png")
    descifrado_imagen_cbc = aes_cbc.descifrar(cifrado_imagen_cbc, iv_cbc)
    guardar_imagen(descifrado_imagen_cbc, "imagen_descifrada_cbc.png")

    print("\n-- Prueba con AES (ECB) para Imagen --")
    aes_ecb = AES_ECB()
    cifrado_imagen_ecb = aes_ecb.cifrar(imagen)
    guardar_imagen(cifrado_imagen_ecb, "imagen_cifrada_ecb.png")
    descifrado_imagen_ecb = aes_ecb.descifrar(cifrado_imagen_ecb)
    guardar_imagen(descifrado_imagen_ecb, "imagen_descifrada_ecb.png")



-- Prueba con AES (CBC) para Imagen --

-- Prueba con AES (ECB) para Imagen --


### Preguntas


¿Qué tamaño de clave se está usando para DES, 3DES y AES?
- DES, usa una clave de 56 bits. Aunque se genera un bloque de 64 bits, solo se usan 56 bits efectivamente y los otros 8 se usan para detección de errores, 3 DES usa 3 claves DES concatenadas, por lo que el tamaño total de la clave es de 168 bits. Sin embargo, la seguridad real es de 112 bits debido a ataques de tipo meet-in-the-middle, AES, se puede usar con claves de 128 bits, 192 bits o 256 bits. Se esta usando 128 bits.

---

¿Qué modo de operación está implementado?
1. DES - ECB.
2. 3DES - CBC.
3. AES -  CBC y ECB.
 
---

¿Por qué no debemos usar ECB en datos sensibles?
- El modo ECB es inseguro porque cifra cada bloque de datos de manera independiente y sin relación con otros bloques, facilita ataques de análisis de patrones y es completamente determinístico: siempre da el mismo resultado para un mismo mensaje con la misma clave.
---

¿Cuál es la diferencia entre ECB vs CBC? ¿Se puede notar directamente en una imagen?

- Se puede notar directamente en una imagen. Al cifrar una imagen con ECB, si la imagen tiene áreas con colores sólidos o patrones repetitivos, esos patrones aparecerán en la imagen cifrada. Con CBC, no se verá ningún patrón porque cada bloque de datos se mezcla con el anterior, ocultando las similitudes.

---

¿Qué es el IV?
- El IV es un bloque de datos que se usa en algunos modos de cifrado para añadir aleatoriedad al proceso de cifrado. Debe ser único y aleatorio para cada cifrado con la misma clave. En CBC, cada bloque cifrado se combina con el IV o con el bloque cifrado anterior, evitando que bloques idénticos produzcan resultados idénticos.
---

¿Qué es el PADDING?
- El Padding se utiliza cuando los datos a cifrar no son un múltiplo exacto del tamaño del bloque. Si el mensaje no tiene un tamaño que sea múltiplo del tamaño del bloque, se añade un relleno al final para completarlo. 

---

¿En qué situaciones se recomienda cada modo de operación?
- ECB - Solo si no hay datos sensibles y no hay riesgo de análisis de patrones. 
- CBC - Uso general cuando se requiere seguridad de alto nivel, especialmente con datos que tienen patrones claros. 
- CTR / GCM / CCM - Modos recomendados para datos en movimiento o comunicación de red, ya que permiten cifrado en paralelo y autenticación de datos.

---

¿Cómo elegir un modo seguro en cada lenguaje de programación?
- Usar bibliotecas seguras y probadas, como:
- Python `pycryptodome` (Modos recomendados: `CBC`, `GCM`), java `javax.crypto` (Modos recomendados: `CBC`, `GCM`), c#  `System.Security.Cryptography` (Modos recomendados: `GCM`, `CBC`)

Recomendaciones generales:
  - Evitar **ECB** en datos sensibles.
  - Preferir **GCM o CTR** si se necesita cifrado paralelo o autenticado.
  - Siempre generar un **IV aleatorio** para cada cifrado.
  - Usar **llaves aleatorias y largas**.